# Exploratory Data Analysis

In [ ]:
import pickle
import os
import nibabel as nib
import numpy as np
import math
import torchio as tio
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import random
import matplotlib.patches as patch
from safetensors import safe_open
from pathlib import Path
from glob import glob

from med_slim.utils.preprocessing.transforms import get_transforms, get_adaptive_transform
from med_slim.data.slice_dataset import (
    TiledSliceDataset,
    build_pre_tile_transform,
    generate_tiled_images,
)
from med_slim.utils.preprocessing.precompute_slice_feature import build_transform
from med_slim.utils.model_config import get_slice_encoder_config

In [ ]:
VIEW_PLANES = ['sagittal', 'coronal', 'axial']

In [ ]:
def load_nifti_image(data_root_dir, exam_id, view_plane):
    nifti_file_path = f'{data_root_dir}/{view_plane}/{exam_id}.nii.gz'
    return nib.load(nifti_file_path).get_fdata()

In [ ]:
def get_num_images_per_exam(exams_path):
    num_slices_per_exam = []
    for exam in exams_path:
        with safe_open(exam, framework='pt', device='cpu') as sf:
            feats = sf.get_tensor('feats')
            num_slices_per_exam.append(feats.shape[0])   
    return np.asarray(num_slices_per_exam)

In [ ]:
# Check raw slice counts from precomputed features
def plot_num_slices_per_exam(dataset_path):
    f, axes = plt.subplots(1, 3, figsize=(20, 6), sharex=True)
    for i, plane in enumerate(VIEW_PLANES):
        feat_dir = f'{dataset_path}/{plane}'
        if not os.path.exists(feat_dir):
            print(f'{plane} plane is not provided in this dataset.')
            continue
        files = glob(f'{feat_dir}/*.safetensors')
        slices = get_num_images_per_exam(files)
        print(f'{plane} (N={len(slices)}): min={min(slices)}, max={max(slices)}, mean={np.mean(slices):.1f}, median={np.median(slices):.0f}, std={np.std(slices):.1f}')
        sns.histplot(slices, ax=axes[i], kde=True)
        axes[i].set_title(f'{plane} plane')

In [ ]:
def get_inplane_resolution(data_root_dir, view_plane, n=500):
    files = glob(f"{data_root_dir}/{view_plane}/*.nii.gz")
    samples = random.sample(files, n)
    if files:
        shapes = []
        spacings = []
        for f in samples:
            img = nib.load(f)
            shapes.append(img.shape)
            spacings.append(img.header.get_zooms())
        shapes = np.array(shapes)
        spacings = np.array(spacings)
        avg_inplane = np.mean(shapes, axis=0)
        avg_inplane = np.rint(avg_inplane).astype(int)
        avg_spacing = np.round(np.mean(spacings, axis=0), 3)
        print(f'{view_plane}: Avg in-plane resolution {avg_inplane}')
        print(f'{view_plane}: Avg spacing {avg_spacing}')
    else:
        raise FileNotFoundError

In [ ]:
def region_labels(grid_size: int):
    return ["global"] + [f"r{r}c{c}" for r in range(grid_size) for c in range(grid_size)]
def to_display(img2d: np.ndarray) -> np.ndarray:
    """Robust grayscale scaling for MRI display only (not model input)."""
    x = img2d.astype(np.float32)
    lo, hi = np.percentile(x, [1, 99])
    x = np.clip(x, lo, hi)
    return (x - lo) / (hi - lo + 1e-8)
def slice_2d(volume_cwhd: torch.Tensor, slice_idx: int) -> np.ndarray:
    """Return display array from TorchIO tensor [C, W, H, D] with shape (H, W): height on y, width on x."""
    wh = volume_cwhd[0, :, :, slice_idx].cpu().numpy()  # (W, H)
    return wh.T
def overlay_tiles(ax, region_boxes, W, H, labels=None, linewidth=2):
    """Draw normalized region_boxes on an imshow with array shape (W, H)."""
    for i, box in enumerate(region_boxes):
        w0n, h0n, w1n, h1n = box
        w0, w1 = w0n * W, w1n * W
        h0, h1 = h0n * H, h1n * H
        color = "yellow" if i == 0 else "cyan"
        rect = patch.Rectangle(
            (w0, h0), w1 - w0, h1 - h0, 
            fill=False, edgecolor=color, linewidth=linewidth,
        )
        ax.add_patch(rect)
        if labels is not None:
            if i == 0:
                ax.text(W // 2 - 15, H - 10, labels[i], color=color, fontsize=9, weight="bold")
            else:
                ax.text(w0 + 2, h0 + 2, labels[i], color=color, fontsize=9, weight="bold")

def show_slice(ax, volume_cwhd, slice_idx, title="", region_boxes=None, labels=None):
    W, H = volume_cwhd.shape[1], volume_cwhd.shape[2]
    img = slice_2d(volume_cwhd, slice_idx)  # (H, W)
    ax.imshow(
        to_display(img),
        cmap="gray",
        origin="lower",
        extent=[0, W, 0, H],   # x=width, y=height
        aspect="equal",
    )
    if region_boxes is not None:
        overlay_tiles(ax, region_boxes, W, H, labels)
    ax.set_title(title)
    ax.set_xlabel("width (W)")
    ax.set_ylabel("height (H)")

## MRNet

### Check in-plane resolution distribution

In [ ]:
get_inplane_resolution('/hpcwork/rwth1833/datasets/preprocessed/MRNet/train', VIEW_PLANES[0], n=100)

In [ ]:
get_inplane_resolution('/hpcwork/rwth1833/datasets/preprocessed/MRNet/train', VIEW_PLANES[1], n=100)

In [ ]:
get_inplane_resolution('/hpcwork/rwth1833/datasets/preprocessed/MRNet/train', VIEW_PLANES[2], n=100)

In [ ]:
get_inplane_resolution('/hpcwork/rwth1833/datasets/preprocessed/MRNet/test', VIEW_PLANES[0], n=100)

In [ ]:
get_inplane_resolution('/hpcwork/rwth1833/datasets/preprocessed/MRNet/test', VIEW_PLANES[1], n=100)

In [ ]:
get_inplane_resolution('/hpcwork/rwth1833/datasets/preprocessed/MRNet/test', VIEW_PLANES[2], n=100)

### Before `torchio.transforms`

In [ ]:
data_root_dir = '/hpcwork/rwth1833/datasets/preprocessed/MRNet/train'
exam_id = '1129'

In [ ]:
# Access center slices along different axes (e.g., axial, sagittal, coronal)
fig, axes = plt.subplots(1, 3, figsize=(10, 5))
for i, slice_view in enumerate(VIEW_PLANES):
    image = load_nifti_image(data_root_dir, exam_id, slice_view)
    slice_idx = image.shape[-1] // 2
    axes[i].imshow(image[:, :, slice_idx], cmap='gray', origin='lower')
    axes[i].set_title(f'{slice_view} slice')
    axes[i].axis('off')
plt.show()

In [ ]:
# Load an image
image = load_nifti_image(data_root_dir, exam_id, VIEW_PLANES[0])
image.shape

In [ ]:
slice_idx = image.shape[-1] // 2
plt.imshow(image[:, :, slice_idx], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
num_slices = image.shape[-1]
n_cols = 8
n_rows = math.ceil(num_slices / n_cols)

plt.figure(figsize=(n_cols * 2, n_rows * 2))
for i in range(num_slices):
    plt.subplot(n_rows, n_cols, i + 1)
    plt.imshow(image[:, :,i], cmap='gray')
    plt.axis('off')
plt.tight_layout()
plt.show()

### After `torchio.transforms`

#### Resize transform

In [ ]:
_, val_tf = get_transforms(model_name="dinov2", plane=VIEW_PLANES[0], spatial_mode="resize", to_tensor=True)

In [ ]:
nifti_file_path = f'{data_root_dir}/{VIEW_PLANES[0]}/{exam_id}.nii.gz'
original_image = tio.ScalarImage(nifti_file_path)
print(f"Original image shape: {original_image.shape}")
print(f"Original image spacing: {original_image.spacing}")

In [ ]:
transformed_image = val_tf(original_image)
print(f"Transformed image shape: {transformed_image.shape}")

In [ ]:
transformed_image = transformed_image.numpy().squeeze(axis=0) # Remove channel dimension for visualization 
slice_idx = transformed_image.shape[0] // 2
plt.imshow(transformed_image[slice_idx], cmap='gray')
plt.axis('off')
plt.show()

#### Adaptive transform based on dataset in-plane resolution and pretrained FM required resolution: 
In this case, CropOrPad

In [ ]:
adaptive_tf = get_adaptive_transform(model_name="dinov2", plane=VIEW_PLANES[0])
nifti_file_path = f'{data_root_dir}/{VIEW_PLANES[0]}/{exam_id}.nii.gz'
original_image = tio.ScalarImage(nifti_file_path)
print(f"Original image shape: {original_image.shape}")
print(f"Original image spacing: {original_image.spacing}")

In [ ]:
transformed_image = adaptive_tf(original_image)
print(f"Transformed image shape: {transformed_image.shape}")
transformed_image = transformed_image.numpy().squeeze(axis=0) # Remove channel dimension for visualization 
slice_idx = transformed_image.shape[0] // 2
plt.imshow(transformed_image[slice_idx], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
num_slices = transformed_image.shape[0]
n_cols = 8
n_rows = math.ceil(num_slices / n_cols)

plt.figure(figsize=(n_cols * 2, n_rows * 2))
for i in range(num_slices):
    plt.subplot(n_rows, n_cols, i + 1)
    plt.imshow(transformed_image[i], cmap='gray')
    plt.axis('off')
plt.tight_layout()
plt.show()

### Slice Count distribution

In [ ]:
feat_dir = '/hpcwork/rwth1833/feat_caches/MRNet/slices_raw/crop/dinov2/train/'
plot_num_slices_per_exam(feat_dir)

In [ ]:
feat_dir = '/hpcwork/rwth1833/feat_caches/MRNet/slices_raw/crop/dinov2/test/'
plot_num_slices_per_exam(feat_dir)

### Cropped tile regions

In [ ]:
DATA_DIR = "/hpcwork/rwth1833/datasets/preprocessed/MRNet"
SPLIT = "train"
PLANE = "axial"
SPATIAL_MODE = "crop"
REGIONAL_TOKENS = 4          # 2x2 regional + 1 global
MODEL_NAME = "dinov2"        # pick any FM you plan to precompute
CROP_EMPTY_SLICES = True
USE_RAW_SLICE_RESOLUTION = True

GRID_SIZE = int(math.isqrt(REGIONAL_TOKENS))
assert GRID_SIZE * GRID_SIZE == REGIONAL_TOKENS
REGION_LABELS = ["global"] + [f"r{r}c{c}" for r in range(GRID_SIZE) for c in range(GRID_SIZE)]
print("Regions:", REGION_LABELS)

In [ ]:
# Build transforms
pre_tile_transform = build_pre_tile_transform(
    plane=PLANE,
    crop_empty_slices=CROP_EMPTY_SLICES,
)
view_transform = build_transform(
    model_name=MODEL_NAME,
    spatial_mode=SPATIAL_MODE,
    plane=PLANE,
    num_slices=None if USE_RAW_SLICE_RESOLUTION else 32,
    crop_empty_slices=False,  # already handled in pre_tile when enabled
)
ds = TiledSliceDataset(
    path_root=DATA_DIR,
    split=SPLIT,
    pre_tile_transform=pre_tile_transform,
    view_transform=view_transform,
    grid_size=GRID_SIZE,
    plane=PLANE,
)
sample_idx = random.randrange(len(ds))
uid = ds.sample_ids[sample_idx]
print(f"Sample index={sample_idx}, uid={uid}, dataset size={len(ds)}")

item = ds[sample_idx]
fm_views = item["source"]              # [num_regions, C, W_fm, H_fm, D]
region_boxes = item["region_boxes"]    # normalized boxes, len = 1 + grid^2
print("FM view tensor:", tuple(fm_views.shape))
print("region_boxes[0] (global):", region_boxes[0])
raw_sample = pre_tile_transform(tio.ScalarImage(ds.get_nifti_path(uid)))
native_images, native_boxes = generate_tiled_images(raw_sample, GRID_SIZE)
num_slices = raw_sample.tensor.shape[-1]
slice_idx = num_slices // 2
print(f"Native volume shape (C,W,H,D): {tuple(raw_sample.tensor.shape)}, slice_idx={slice_idx}")
fm_cfg = get_slice_encoder_config(MODEL_NAME)
print(f"FM target in-plane size (H, W): {fm_cfg['img_size']}")

In [ ]:
W, H = raw_sample.tensor.shape[1], raw_sample.tensor.shape[2]
global_native = slice_2d(raw_sample.tensor, slice_idx)

# (1) Global native slice and tile grid
fig, ax = plt.subplots(1, 1)
show_slice(
    ax,
    raw_sample.tensor,
    slice_idx,
    title="Native Global + Cropped Tiles",
    region_boxes=native_boxes,
    labels=REGION_LABELS,
)
plt.tight_layout()
plt.show()

In [ ]:
# (2) Native crops (before adaptive FM transform)
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
axes = axes.flatten()  # 1D list of 4 Axes
for i, (img, ax) in enumerate(zip(native_images[1:], axes), start=1):
    show_slice(ax, img.tensor, slice_idx, title=f"Native crop: {REGION_LABELS[i]}")
plt.tight_layout()
plt.show()

In [ ]:
# (3) FM-ready views after adaptive transform (what encoder receives)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()
for i, ax in enumerate(axes):
    if i >= fm_views.shape[0]:
        ax.axis("off")
        continue
    show_slice(ax, fm_views[i], slice_idx, title=f"FM input: {REGION_LABELS[i]}")
plt.tight_layout()
plt.show()

In [ ]:
# Sanity checks
assert len(region_boxes) == 1 + GRID_SIZE ** 2
assert region_boxes == native_boxes
print("\nTile box fractions (w0/W, h0/H, w1/W, h1/H):")
for label, box in zip(REGION_LABELS, region_boxes):
    print(f"  {label:8s}: {box}")
# Fraction of in-plane area covered by regional tiles (should be ~1.0)
regional = native_boxes[1:]
coverage = sum((b[2]-b[0]) * (b[3]-b[1]) for b in regional)
print(f"Regional tile area fraction (should ~= 1.0): {coverage:.4f}")

## kneeMRI

#### Raw data

In [ ]:
# directory where the volumetric data is located
volumetric_data_dir = '/hpcwork/rwth1833/datasets/kneeMRI/vol_train'

# path to metadata csv file
metadata_csv_path = '/hpcwork/rwth1833/datasets/kneeMRI/metadata.csv'

# names=True loads the interprets the first row of csv file as column names
# 'i4' = 4 byte signed integer, 'U20' = unicode max 20 char string
metadata = np.genfromtxt(metadata_csv_path, delimiter=',', names=True, 
    dtype='i4,i4,i4,i4,i4,i4,i4,i4,i4,i4,U20') 

print('Column names:')
print(metadata.dtype.names)

# Select the specified examID
exams = metadata[metadata['examId'] == 518793]


In [ ]:
for exam in exams:
    vol_data_file = exam['volumeFilename']

    vol_data_path = os.path.join(volumetric_data_dir, vol_data_file)

    # Load data from file
    with open(vol_data_path, 'rb') as file_handler: # Must use 'rb' as the data is binary
        volumetric_data = pickle.load(file_handler)
    
    print('\nShape of volume "%s":' % vol_data_path, volumetric_data.shape)
    
    # Get all roi slices from volume
    z_start = exam['roiZ']
    depth = exam['roiDepth']
    
    for z in range(z_start, z_start + depth):
    
        slice = volumetric_data[z, :, :]
        
        # Get roi dimensions
        x, y, w, h = [exam[attr] for attr in ['roiX', 'roiY', 'roiWidth', 'roiHeight']]
        
        # Extract ROI
        roi = slice[y:y+h, x:x+w]
        
        # Plot slice and roi
        figure = plt.figure()
        plot = plt.subplot2grid((1, 4), (0, 0), 1, 3) # This makes the slice plot larger than roi plot
        plot.add_patch(patch.Rectangle((x, y), w, h, fill=None, color='red'))
        plot.imshow(slice, cmap='gray')
        plot = plt.subplot2grid((1, 4), (0, 3), 1, 1)
        plot.imshow(roi, cmap='gray')
        
        plt.show()

In [ ]:
data_root_dir = '/hpcwork/rwth1833/datasets/preprocessed/kneeMRI/train'
image = load_nifti_image(data_root_dir,"518793-5", VIEW_PLANES[0])
image.shape

In [ ]:
slice_idx = image.shape[-1] // 2
plt.imshow(image[:, :, slice_idx], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
num_slices = image.shape[-1]
n_cols = 8
n_rows = math.ceil(num_slices / n_cols)

plt.figure(figsize=(n_cols * 2, n_rows * 2))
for i in range(num_slices):
    plt.subplot(n_rows, n_cols, i + 1)
    plt.imshow(image[:, :,i], cmap='gray')
    plt.axis('off')
plt.tight_layout()
plt.show()

#### Check in-plane resolution

In [ ]:
get_inplane_resolution('/hpcwork/rwth1833/datasets/preprocessed/kneeMRI/train', VIEW_PLANES[0], n=100)

In [ ]:
get_inplane_resolution('/hpcwork/rwth1833/datasets/preprocessed/kneeMRI/test', VIEW_PLANES[0], n=50)

#### After `torchio.transform`

In [ ]:
exam_id = "518793-5"
_, val_tf = get_transforms(model_name="dinov2", plane=VIEW_PLANES[0], spatial_mode="crop", to_tensor=True)
nifti_file_path = f'{data_root_dir}/{VIEW_PLANES[0]}/{exam_id}.nii.gz'
original_image = tio.ScalarImage(nifti_file_path)
print(f"Original image shape: {original_image.shape}")
print(f"Original image spacing: {original_image.spacing}")
transformed_image = val_tf(original_image)
print(f"Transformed image shape: {transformed_image.shape}")

In [ ]:
transformed_image = transformed_image.numpy().squeeze(axis=0) # Remove channel dimension for visualization 
slice_idx = transformed_image.shape[0] // 2
plt.imshow(transformed_image[slice_idx], cmap='gray')
plt.axis('off')
plt.show()

#### Slice counts distribution

In [ ]:
feat_dir = '/hpcwork/rwth1833/feat_caches/kneeMRI/slices_raw/crop/dinov2/train/'
plot_num_slices_per_exam(feat_dir)

In [ ]:
feat_dir = '/hpcwork/rwth1833/feat_caches/kneeMRI/slices_raw/crop/dinov2/test/'
plot_num_slices_per_exam(feat_dir)

## fastMRI

### In-plane resolution distribution

In [ ]:
for view_plane in VIEW_PLANES:
    get_inplane_resolution('/hpcwork/rwth1833/datasets/preprocessed/fastMRI/train', view_plane, n=100)

### Before `torchio.transforms`

In [ ]:
data_root_dir = '/hpcwork/rwth1833/datasets/preprocessed/fastMRI/train/pd'
view_plane = VIEW_PLANES[1]
exam_id = 'study_42f9ce23_MR5_bae0cac8'
image = load_nifti_image(data_root_dir, exam_id, view_plane)
image.shape

In [ ]:
slice_idx = image.shape[-1] // 2
plt.imshow(image[:, :, slice_idx], cmap='gray')
plt.axis('off')
plt.show()

### After `torchio.transforms`

In [ ]:
adaptive_tf = get_adaptive_transform(model_name="ark", plane=VIEW_PLANES[1])
nifti_file_path = f'{data_root_dir}/{VIEW_PLANES[1]}/{exam_id}.nii.gz'
original_image = tio.ScalarImage(nifti_file_path)
print(f"Original image shape: {original_image.shape}")
print(f"Original image spacing: {original_image.spacing}")
transformed_image = adaptive_tf(original_image)
print(f"Transformed image shape: {transformed_image.shape}")
transformed_image = transformed_image.numpy().squeeze(axis=0) # Remove channel dimension for visualization 
slice_idx = transformed_image.shape[0] // 2
plt.imshow(transformed_image[slice_idx], cmap='gray')
plt.axis('off')
plt.show()

### Before `torchio.transforms`

In [ ]:
data_root_dir = '/hpcwork/rwth1833/datasets/preprocessed/fastMRI/train/'
view_plane = VIEW_PLANES[2]
exam_id = 'study_39a470be_MR4_21d1f671'
image = load_nifti_image(data_root_dir, exam_id, view_plane)
image.shape

In [ ]:
slice_idx = image.shape[-1] // 2
plt.imshow(image[:, :, slice_idx], cmap='gray')
plt.axis('off')
plt.show()

### After `torchio.transforms`

In [ ]:
adaptive_tf = get_adaptive_transform(model_name="rad-dino", plane=VIEW_PLANES[2])
nifti_file_path = f'{data_root_dir}/{VIEW_PLANES[2]}/{exam_id}.nii.gz'
original_image = tio.ScalarImage(nifti_file_path)
print(f"Original image shape: {original_image.shape}")
print(f"Original image spacing: {original_image.spacing}")
transformed_image = adaptive_tf(original_image)
print(f"Transformed image shape: {transformed_image.shape}")
transformed_image = transformed_image.numpy().squeeze(axis=0) # Remove channel dimension for visualization 
slice_idx = transformed_image.shape[0] // 2
plt.imshow(transformed_image[slice_idx], cmap='gray')
plt.axis('off')
plt.show()

### Before `torchio.transforms`

In [ ]:
data_root_dir = '/hpcwork/rwth1833/datasets/preprocessed/fastMRI/train/'
view_plane = VIEW_PLANES[1]
exam_id = 'study_16dabf31_MR10_93ddb94c'
image = load_nifti_image(data_root_dir, exam_id, view_plane)
image.shape

In [ ]:
slice_idx = image.shape[-1] // 2
plt.imshow(image[:, :, slice_idx], cmap='gray')
plt.axis('off')
plt.show()

### After `torchio.transforms`

In [ ]:
_, val_tf = get_transforms(model_name="dinov3", plane=VIEW_PLANES[1], spatial_mode="resample", to_tensor=True)
original_image = tio.ScalarImage(f'{data_root_dir}/{view_plane}/{exam_id}.nii.gz')
print(f"Original image shape: {original_image.shape}")
print(f"Original image spacing: {original_image.spacing}")
transformed_image = val_tf(original_image)
print(f"Transformed image shape: {transformed_image.shape}")

In [ ]:
transformed_image = transformed_image.numpy().squeeze(axis=0) # Remove channel dimension for visualization 
plt.imshow(transformed_image[slice_idx], cmap='gray')
plt.axis('off')
plt.show()

### Slcie counts distribution

In [ ]:
feat_dir = '/hpcwork/rwth1833/feat_caches/fastMRI/slices_raw/adaptive/dinov2/train/'
plot_num_slices_per_exam(feat_dir)

### Cropped Tiles

In [ ]:
DATA_DIR = "/hpcwork/rwth1833/datasets/preprocessed/fastMRI"
SPLIT = "train"
PLANE = "coronal"
MRI_SEQUENCES = "all"        # fastMRI: {split}/{sequence}/{plane}/; omit for flat MRNet layout
SPATIAL_MODE = "adaptive"
REGIONAL_TOKENS = 4          # 2x2 regional + 1 global
MODEL_NAME = "ark"        # pick any FM you plan to precompute
CROP_EMPTY_SLICES = True
USE_RAW_SLICE_RESOLUTION = True

GRID_SIZE = int(math.isqrt(REGIONAL_TOKENS))
assert GRID_SIZE * GRID_SIZE == REGIONAL_TOKENS
REGION_LABELS = ["global"] + [f"r{r}c{c}" for r in range(GRID_SIZE) for c in range(GRID_SIZE)]
print("Regions:", REGION_LABELS)

In [ ]:
# Build transforms
pre_tile_transform = build_pre_tile_transform(
    plane=PLANE,
    crop_empty_slices=CROP_EMPTY_SLICES,
)
view_transform = build_transform(
    model_name=MODEL_NAME,
    spatial_mode=SPATIAL_MODE,
    plane=PLANE,
    num_slices=None if USE_RAW_SLICE_RESOLUTION else 32,
    crop_empty_slices=False,  # already handled in pre_tile when enabled
)
ds = TiledSliceDataset(
    path_root=DATA_DIR,
    split=SPLIT,
    pre_tile_transform=pre_tile_transform,
    view_transform=view_transform,
    grid_size=GRID_SIZE,
    plane=PLANE,
    mri_sequences=MRI_SEQUENCES,
)
sample_idx = random.randrange(len(ds))
uid = ds.sample_ids[sample_idx]
print(f"Sample index={sample_idx}, uid={uid}, dataset size={len(ds)}")

item = ds[sample_idx]
fm_views = item["source"]              # [num_regions, C, W_fm, H_fm, D]
region_boxes = item["region_boxes"]    # normalized boxes, len = 1 + grid^2
print("FM view tensor:", tuple(fm_views.shape))
print("region_boxes[0] (global):", region_boxes[0])
raw_sample = pre_tile_transform(tio.ScalarImage(ds.get_nifti_path(uid)))
native_images, native_boxes = generate_tiled_images(raw_sample, GRID_SIZE)
num_slices = raw_sample.tensor.shape[-1]
slice_idx = num_slices // 2
print(f"Native volume shape (C,W,H,D): {tuple(raw_sample.tensor.shape)}, slice_idx={slice_idx}")
fm_cfg = get_slice_encoder_config(MODEL_NAME)
print(f"FM target in-plane size (H, W): {fm_cfg['img_size']}")

In [ ]:
W, H = raw_sample.tensor.shape[1], raw_sample.tensor.shape[2]
global_native = slice_2d(raw_sample.tensor, slice_idx)

# (1) Global native slice and tile grid
fig, ax = plt.subplots(1, 1)
show_slice(
    ax,
    raw_sample.tensor,
    slice_idx,
    title="Native Global + Cropped Tiles",
    region_boxes=native_boxes,
    labels=REGION_LABELS,
)
plt.tight_layout()
plt.show()

## KMAR-50K

In [ ]:
for view_plane in VIEW_PLANES:
    get_inplane_resolution('/hpcwork/rwth1833/datasets/preprocessed/KMAR-50K/train', view_plane, n=100)

### Before `torchio.transforms`

In [ ]:
nifti_file_path = '/hpcwork/rwth1833/datasets/preprocessed/KMAR-50K/train/sagittal/2021_272_MR0.nii.gz'
image = nib.load(nifti_file_path).get_fdata()
image.shape

In [ ]:
slice_idx = image.shape[-1] // 2
plt.imshow(image[:, :, slice_idx], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
num_slices = image.shape[-1]
n_cols = 8
n_rows = math.ceil(num_slices / n_cols)

plt.figure(figsize=(n_cols * 2, n_rows * 2))
for i in range(num_slices):
    plt.subplot(n_rows, n_cols, i + 1)
    plt.imshow(image[:, :,i], cmap='gray')
    plt.axis('off')
plt.tight_layout()
plt.show()

### After `torchio.transforms`

In [ ]:
adaptive_tf = get_adaptive_transform(model_name="biomedclip", plane=VIEW_PLANES[0])

In [ ]:
original_image = tio.ScalarImage(nifti_file_path)
print(f"Original image shape: {original_image.shape}")
print(f"Original image spacing: {original_image.spacing}")
transformed_image = adaptive_tf(original_image)
print(f"Transformed image shape: {transformed_image.shape}")

In [ ]:
transformed_image = transformed_image.numpy().squeeze(axis=0) # Remove channel dimension for visualization 
plt.imshow(transformed_image[slice_idx], cmap='gray')
plt.axis('off')
plt.show()

### Slice counts distribution

In [ ]:
feat_dir = '/hpcwork/rwth1833/feat_caches/KMAR-50K/slices_raw/adaptive/dinov2/train/'
plot_num_slices_per_exam(feat_dir)

## SKM-TEA

In [ ]:
for split in ['DESS_E1/train', 'DESS_E1/val', 'DESS_E1/test', 'DESS_E2/train', 'DESS_E2/val', 'DESS_E2/test']:
    get_inplane_resolution(f'/hpcwork/rwth1833/datasets/preprocessed/SKM-TEA/{split}', VIEW_PLANES[0], n=25)

### Before `torchio.transforms`

In [ ]:
nifti_file_path = '/hpcwork/rwth1833/datasets/preprocessed/SKM-TEA/DESS_E1/train/sagittal/MTR_094.nii.gz'
image = nib.load(nifti_file_path).get_fdata()
image.shape

In [ ]:
slice_idx = image.shape[-1] // 2
plt.imshow(image[:, :, slice_idx], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
nifti_file_path = '/hpcwork/rwth1833/datasets/preprocessed/SKM-TEA/DESS_E2/train/sagittal/MTR_094.nii.gz'
image = nib.load(nifti_file_path).get_fdata()
slice_idx = image.shape[-1] // 2
plt.imshow(image[:, :, slice_idx], cmap='gray')
plt.axis('off')
plt.show()

### After `torchio.transforms`

In [ ]:
adaptive_tf = get_adaptive_transform(model_name="ark", plane=VIEW_PLANES[0])
original_image = tio.ScalarImage(nifti_file_path)
print(f"Original image shape: {original_image.shape}")
print(f"Original image spacing: {original_image.spacing}")
transformed_image = adaptive_tf(original_image)
print(f"Transformed image shape: {transformed_image.shape}")

In [ ]:
transformed_image = transformed_image.numpy().squeeze(axis=0) # Remove channel dimension for visualization 
plt.imshow(transformed_image[slice_idx], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
for seq_type in ['DESS_E1', 'DESS_E2']:
    for split in ['train', 'val', 'test']: 
        feat_dir = f'/hpcwork/rwth1833/feat_caches/SKM-TEA/{seq_type}/slices_raw/adaptive/dinov2/{split}/'
        plot_num_slices_per_exam(feat_dir)

## LIDC-IDRI

In [ ]:
nifti_file_path = '/hpcwork/rwth1833/datasets/preprocessed/LIDC-IDRI/909.nii.gz'
image = nib.load(nifti_file_path).get_fdata()
image.shape

In [ ]:
slice_idx = image.shape[-1] // 2
plt.imshow(image[:, :, slice_idx], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
original_image = tio.ScalarImage(nifti_file_path)
adaptive_tf = get_adaptive_transform(model_name="medsiglip", plane=VIEW_PLANES[2])
transformed_image = adaptive_tf(original_image)
transformed_image = transformed_image.numpy().squeeze(axis=0) # Remove channel dimension for visualization 
plt.imshow(transformed_image[slice_idx], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
transformed_image.shape

## DukeBreastMRI

In [ ]:
nifti_file_path = '/hpcwork/rwth1833/datasets/preprocessed/DukeBreastMRI/post/train/axial/Breast_MRI_673.nii.gz'
image = nib.load(nifti_file_path).get_fdata()
image.shape

In [ ]:
slice_idx = image.shape[-1] // 2
plt.imshow(image[:, :, slice_idx], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
nifti_file_path = '/hpcwork/rwth1833/datasets/preprocessed/DukeBreastMRI/pre/train/axial/Breast_MRI_673.nii.gz'
image = nib.load(nifti_file_path).get_fdata()
image.shape

In [ ]:
slice_idx = image.shape[-1] // 2
plt.imshow(image[:, :, slice_idx], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
nifti_file_path = '/hpcwork/rwth1833/datasets/preprocessed/DukeBreastMRI/subtracted/train/axial/Breast_MRI_673.nii.gz'
image = nib.load(nifti_file_path).get_fdata()
image.shape

In [ ]:
slice_idx = image.shape[-1] // 2
plt.imshow(image[:, :, slice_idx], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
original_image = tio.ScalarImage(nifti_file_path)
adaptive_tf = get_adaptive_transform(model_name="ark", plane=VIEW_PLANES[2])
transformed_image = adaptive_tf(original_image)
transformed_image = transformed_image.numpy().squeeze(axis=0) # Remove channel dimension for visualization 
plt.imshow(transformed_image[slice_idx], cmap='gray')
plt.axis('off')
plt.show()

## NLST

In [ ]:
nifti_file_path = '/hpcwork/rwth1833/datasets/preprocessed/NLST/215216_1.nii.gz'
image = nib.load(nifti_file_path).get_fdata()
print(image.shape)
slice_idx = image.shape[-1] // 2
plt.imshow(image[:, :, slice_idx], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
original_image = tio.ScalarImage(nifti_file_path)
adaptive_tf = get_adaptive_transform(model_name="medsiglip", plane=VIEW_PLANES[2])
transformed_image = adaptive_tf(original_image)
transformed_image = transformed_image.numpy().squeeze(axis=0) # Remove channel dimension for visualization 
plt.imshow(transformed_image[slice_idx], cmap='gray')
plt.axis('off')
plt.show()

## LUNA25

In [ ]:
nifti_file_path = '/hpcwork/rwth1833/datasets/preprocessed/LUNA25/1.3.6.1.4.1.14519.5.2.1.7009.9004.782097239773095151157864320321.nii.gz'
image = nib.load(nifti_file_path).get_fdata()
print(image.shape)
slice_idx = image.shape[-1] // 2
plt.imshow(image[:, :, slice_idx], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
original_image = tio.ScalarImage(nifti_file_path)
adaptive_tf = get_adaptive_transform(model_name="medsiglip", plane=VIEW_PLANES[2])
transformed_image = adaptive_tf(original_image)
transformed_image = transformed_image.numpy().squeeze(axis=0) # Remove channel dimension for visualization 
plt.imshow(transformed_image[slice_idx], cmap='gray')
plt.axis('off')
plt.show()

## BRAT24

In [ ]:
MRI_SEQ_TYPES = ['t1c', 't1n', 't2w', 't2f']
for seq_type in MRI_SEQ_TYPES:
    get_inplane_resolution('/hpcwork/rwth1833/datasets/preprocessed/BRAT24', seq_type, n=100)

In [ ]:
nifti_file_path = '/hpcwork/rwth1833/datasets/preprocessed/BRAT24/t2f/train/axial/BraTS-GLI-00005-100.nii.gz'
img = nib.load(nifti_file_path).get_fdata()
print(f'Shape: {img.shape}')
print(f'  Axis 0 (R-L): {img.shape[0]} voxels')
print(f'  Axis 1 (A-P): {img.shape[1]} voxels') 
print(f'  Axis 2 (I-S): {img.shape[2]} voxels')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Axial slice: fix axis 2 (I-S), view R-L x A-P
ax_slice = img[:, :, img.shape[2]//2]
axes[0].imshow(ax_slice.T, cmap='gray', origin='lower')
axes[0].set_title('Axial')

# Coronal slice: fix axis 1 (A-P), view R-L x I-S
cor_slice = img[:, img.shape[1]//2, :]
axes[1].imshow(cor_slice.T, cmap='gray', origin='lower')
axes[1].set_title('Coronal')

# Sagittal slice: fix axis 0 (R-L), view A-P x I-S
sag_slice = img[img.shape[0]//2, :, :]
axes[2].imshow(sag_slice.T, cmap='gray', origin='lower')
axes[2].set_title('Sagittal')

In [ ]:
nifti_file_path = '/hpcwork/rwth1833/datasets/preprocessed/BRAT24/t2f/train/axial/BraTS-GLI-00005-100.nii.gz'
image = nib.load(nifti_file_path).get_fdata()
print(image.shape)
slice_idx = image.shape[-1] // 2
plt.imshow(image[:, :, slice_idx], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
original_image = tio.ScalarImage(nifti_file_path)
adaptive_tf = get_adaptive_transform(model_name="dinov2", plane="axial", crop_empty_slices=True, to_tensor=True)
transformed_image = adaptive_tf(original_image)
print(transformed_image.shape)
transformed_image = transformed_image.numpy().squeeze(axis=0) # Remove channel dimension for visualization 
plt.imshow(transformed_image[slice_idx], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
num_slices = transformed_image.shape[0]
n_cols = 15
n_rows = math.ceil(num_slices / n_cols)

plt.figure(figsize=(n_cols * 2, n_rows * 2))
for i in range(num_slices):
    plt.subplot(n_rows, n_cols, i + 1)
    plt.imshow(transformed_image[i], cmap='gray')
    plt.axis('off')
plt.tight_layout()
plt.show()

## CT-RATE

In [ ]:
nifti_file_path = '/hpcwork/rwth1833/datasets/preprocessed/CT-RATE/val/axial/valid_2_a_1.nii.gz'
image = nib.load(nifti_file_path).get_fdata()
print(image.shape)
slice_idx = image.shape[-1] // 2
plt.imshow(image[:, :, slice_idx], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
original_image = tio.ScalarImage(nifti_file_path)
adaptive_tf = get_adaptive_transform(model_name="ark", plane=VIEW_PLANES[2])
transformed_image = adaptive_tf(original_image)
transformed_image = transformed_image.numpy().squeeze(axis=0) # Remove channel dimension for visualization 
plt.imshow(transformed_image[slice_idx], cmap='gray')
plt.axis('off')
plt.show()

## OASIS

In [ ]:
nifti_file_path = '/hpcwork/rwth1833/datasets/preprocessed/CT-RATE/train/axial/train_11347_a_1.nii.gz'
image = nib.load(nifti_file_path).get_fdata()
print(image.shape)
slice_idx = image.shape[-1] // 2
plt.imshow(image[:, :, slice_idx], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
original_image = tio.ScalarImage(nifti_file_path)
adaptive_tf = get_adaptive_transform(model_name="ark", plane=VIEW_PLANES[2])
transformed_image = adaptive_tf(original_image)
transformed_image = transformed_image.numpy().squeeze(axis=0) # Remove channel dimension for visualization 
plt.imshow(transformed_image[slice_idx], cmap='gray')
plt.axis('off')
plt.show()

## RSNA-Brain

In [ ]:
nifti_file_path = '/hpcwork/rwth1833/datasets/preprocessed/RSNA_Brain_Tumor_Radiogenomic/00799/T1w.nii.gz'
image = nib.load(nifti_file_path).get_fdata()
print(image.shape)
slice_idx = image.shape[-1] // 2
plt.imshow(image[:, :, slice_idx], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
original_image = tio.ScalarImage(nifti_file_path)
adaptive_tf = get_adaptive_transform(model_name="ark", plane=VIEW_PLANES[2])
transformed_image = adaptive_tf(original_image)
transformed_image = transformed_image.numpy().squeeze(axis=0) # Remove channel dimension for visualization 
plt.imshow(transformed_image[slice_idx], cmap='gray')
plt.axis('off')
plt.show()